# 04 — Soft segmentation via guided filter

For each stroke detected by the winning hard-segmentation strategy (from `03_strategy_comparison.ipynb`), compute a per-stroke alpha matte using the He et al. 2010 guided filter (paper §3.3, Eq. 5–6).

Flow per painting:

1. Load the winning strategy's binary edge map `T(x)`.
2. Label connected regions of `~T(x)` → one label per stroke region.
3. For each region, build a trimap (foreground=region, background=other regions, unknown=boundary band).
4. Run `guided_filter` using the painting as guide → soft alpha matte per stroke.
5. Stack all alphas into an `.npz`.

In [ ]:
from __future__ import annotations
import sys, json
from pathlib import Path

import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from tqdm import tqdm
from scipy import ndimage as ndi
from skimage import morphology

NB_DIR = Path.cwd()
sys.path.insert(0, str(NB_DIR))
import dstroke_utils as d

REPO_ROOT = Path('C:/Users/MichelleJacobs/Repos/AIAi')
RAW = REPO_ROOT / 'AIA - AI Tool - Fingerprinting-1' / 'data' / 'raw'
OUT = REPO_ROOT / 'AIA - AI Tool - Fingerprinting-1' / 'outputs' / 'dstroke'

winner_file = OUT / 'winning_strategy.txt'
winner = winner_file.read_text().strip() if winner_file.exists() else 'A'
print('Winning strategy:', winner)

In [ ]:
# Load inference helpers from 03
nb = json.load(open(NB_DIR / '03_strategy_comparison.ipynb'))
exec('\n\n'.join(''.join(c['source']) for c in nb['cells']
                  if c['cell_type']=='code' and 'summary =' not in ''.join(c['source'])
                  and 'rows_vis' not in ''.join(c['source'])
                  and 'winner = df' not in ''.join(c['source'])), globals())

In [ ]:
def hard_mask_for(rgb_256: np.ndarray) -> np.ndarray:
    """Return binary edge map T(x) in [0, 1] for the winning strategy."""
    if winner in ('A', 'B') and winner in models:
        _, t = infer_pix2pix(models[winner], rgb_256)
        return t
    if winner == 'D' and 'D' in models:
        pr = infer_segformer(models['D'], rgb_256)
        return (pr > 0.5).astype(np.float32)
    # fallback: reload from disk for C
    return None

def soft_segment(rgb, hard_edge, radius=8, eps=1e-4, unknown_band=3):
    """Return a list of (alpha_matte, mask_bool) for each connected stroke region."""
    interior = hard_edge < 0.5
    labels, n = ndi.label(interior)
    results = []
    for lid in range(1, n+1):
        mask = labels == lid
        if mask.sum() < 30:
            continue
        trimap = np.full(mask.shape, 0.5, dtype=np.float32)
        eroded = morphology.erosion(mask, morphology.disk(unknown_band))
        trimap[eroded] = 1.0
        outside = morphology.dilation(~mask, morphology.disk(unknown_band))
        trimap[(~mask) & outside] = 0.0
        alpha = d.guided_filter(rgb, trimap, radius=radius, eps=eps)
        alpha = np.clip(alpha, 0, 1)
        results.append({'alpha': alpha, 'mask': mask})
    return results

In [ ]:
# pick one representative painting per artist
examples = []
for artist in ('manet', 'van_gogh', 'rembrandt'):
    art_dir = RAW / artist / 'authenticated'
    if not art_dir.exists(): continue
    p = next(iter(p for ext in ('*.tif','*.tiff','*.png','*.jpg') for p in art_dir.glob(ext)), None)
    if p is None: continue
    examples.append((artist, p))

for artist, p in examples:
    rgb = np.array(Image.open(p).convert('RGB').resize((256,256)), dtype=np.float32) / 255.0
    hard = hard_mask_for(rgb)
    if hard is None: continue
    seg = soft_segment(rgb, hard)
    print(f'{artist}/{p.name}: {len(seg)} soft-segmented strokes')
    # stack + save
    if seg:
        stack = np.stack([s['alpha'] for s in seg[:200]])
        out_npz = OUT / 'soft' / artist / f'{p.stem}.npz'
        out_npz.parent.mkdir(parents=True, exist_ok=True)
        np.savez_compressed(out_npz, alphas=stack)
    # visualize the five largest regions
    seg.sort(key=lambda s: -s['mask'].sum())
    fig, axes = plt.subplots(1, min(5, len(seg))+2, figsize=(2.5*(min(5,len(seg))+2), 2.5))
    axes[0].imshow(rgb); axes[0].set_title(f'{artist} input')
    axes[1].imshow(hard, cmap='gray_r'); axes[1].set_title('hard T(x)')
    for k in range(min(5, len(seg))):
        axes[2+k].imshow(seg[k]['alpha'], cmap='gray_r'); axes[2+k].set_title(f'alpha #{k}')
    for a in axes: a.set_xticks([]); a.set_yticks([])
    plt.show()